In [1]:
! git clone https://github.com/MilyaushaShamsutdinova/AlignScore.git 

fatal: destination path 'AlignScore' already exists and is not an empty directory.


In [2]:
%cd AlignScore

/kaggle/working/AlignScore


In [3]:
%%bash
pip install huggingface_hub[hf_transfer]
export HF_HUB_ENABLE_HF_TRANSFER=1

In [4]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(repo_id="CatFr0g/ruAlignScore", filename="RuAlignScore.ckpt")
path

'/root/.cache/huggingface/hub/models--CatFr0g--ruAlignScore/snapshots/bec0aa9d845e8ec27e9df3b598a1655c7d571636/RuAlignScore.ckpt'

In [5]:
import torch

args = {}
args['model'] = 'DeepPavlov/rubert-base-cased'
args['ckpt_path'] = path
args['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
args['batch_size'] = 16
args['max_length'] = 512
args['threshold'] = 0.5

In [6]:
args

{'model': 'DeepPavlov/rubert-base-cased',
 'ckpt_path': '/root/.cache/huggingface/hub/models--CatFr0g--ruAlignScore/snapshots/bec0aa9d845e8ec27e9df3b598a1655c7d571636/RuAlignScore.ckpt',
 'device': 'cuda:0',
 'batch_size': 16,
 'max_length': 512,
 'threshold': 0.5}

In [7]:
from src.inference import Inferencer
from src.AlignScore import AlignScore

model =  AlignScore(model=args['model'], 
                    batch_size=args['batch_size'], 
                    device=args['device'], 
                    ckpt_path=args['ckpt_path'], 
                    evaluation_mode='nli_sp')
score = model.score(contexts=['hello world.'], claims=['hello world.'])
score

2025-04-29 20:29:23.709183: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745958563.732603    1669 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745958563.739555    1669 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.tra

[0.9824784994125366]

In [8]:
model.model.inference(['hello world.'],['hello world.'])

(tensor([4.3924]), tensor([0.6563]), tensor([[0.9825, 0.0101, 0.0074]]))

In [9]:
import random
from tqdm.autonotebook import tqdm
import os
import json
import random
import re
import pandas as pd
import torch
import transformers
from datasets import load_dataset
import logging
from logging import error, info, debug, warning
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
from random import sample

from generate_data import QA2D,QAnswering,MLMGeneratorWithPairedData,ExtractiveSummarizationGenerator


DATASET_HUGGINGFACE = {
    'snli': ['MilyaShams/snli-ru_10k', 'test'],
    'rufact':['akozlova/RuFacts', 'validation'],
    'multi_nli': ['MilyaShams/multi_nli-ru_10k', 'test'],
    'anli': ['MilyaShams/anli-ru_10k', 'test'],
    'nli_fever': ['MilyaShams/nli_fever-ru_10k', 'test'],
    'vitaminc':['MilyaShams/vitaminc-ru_10k', 'test'],
    'doc_nli':['MilyaShams/doc_nli-ru_10k', 'test'],
    'qqp':['MilyaShams/qqp-ru_10k', 'test'],
    'ru_sts':['MilyaShams/ru-stsbenchmark-sts','test'],
    'sberquad':['MilyaShams/sberquad_10k', 'test'],
    'paws':['MilyaShams/paws-ru_10k', 'test'],
    'sick':['MilyaShams/sick-ru', 'test'],
    'race':['MilyaShams/race-ru_10k', 'test'],
    'ms_marco':['MilyaShams/ms_marco-ru_10k', 'test'],
    'ru_sts':['MilyaShams/ru-stsbenchmark-sts', 'test'],
}

DATASET_CONFIG = {
    'snli': {'task': 'nli', 'text_a': 'premise', 'text_b': 'hypothesis', 'label': 'label', 'huggingface': True},
    'rufact': {'task': 'paraphrase', 'text_a': 'evidence', 'text_b': 'claim', 'label': 'label', 'huggingface':True},
    'multi_nli': {'task': 'nli', 'text_a': 'premise', 'text_b': 'hypothesis', 'label': 'label', 'huggingface': True},
    'anli': {'task': 'nli', 'text_a': 'premise', 'text_b': 'hypothesis', 'label': 'label', 'huggingface': True},
    'nli_fever': {'task': 'fact_checking', 'text_a': 'hypothesis', 'text_b': 'premise', 'label': 'label', 'huggingface': True},
    'vitaminc': {'task': 'fact_checking', 'text_a': 'evidence', 'text_b': 'claim', 'label': 'label', 'huggingface':True},
    'doc_nli': {'task': 'bin_nli', 'text_a': 'premise', 'text_b': 'hypothesis', 'label': 'label', 'huggingface': True},
    'qqp': {'task': 'paraphrase', 'text_a': 'text1', 'text_b': 'text2', 'label': 'label', 'huggingface': True},
    'paws': {'task': 'paraphrase', 'text_a': 'sentence1', 'text_b': 'sentence2', 'label': 'label', 'huggingface': True},
    'sick': {'task': 'sts', 'text_a': 'sentence_A', 'text_b': 'sentence_B', 'label': 'relatedness_score', 'huggingface': True},
    'race': {'task': 'qa', 'text_a': 'article', 'text_b': ['question', 'options'], 'label': 'answer', 'huggingface': True}, #TODO: check
    'ms_marco': {'task': 'qa', 'text_a': 'question', 'text_b': 'passage', 'label': 'label', 'huggingface': True},
    'ru_sts': {'task': 'sts', 'text_a': 'sentence1', 'text_b': 'sentence2', 'label': 'score', 'huggingface': True},
}



class DataGenerator():
    def __init__(self, dataset_names, device = None) -> None:
        self.dataset_names = dataset_names
        self.datasets = dict()
        self.t5_qa = None
        self.t5_tokenizer = None
        if device is None:
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        else:
            self.device = device

        self.load_dataset_from_huggingface()

    def load_dataset_from_huggingface(self):
        bar = tqdm(self.dataset_names, desc="Loading datasets")
        for each_dataset in bar:
            bar.set_description(f"Loading {each_dataset}")
            if DATASET_CONFIG[each_dataset].get('huggingface'):
                self.datasets[each_dataset] = load_dataset(
                    *DATASET_HUGGINGFACE[each_dataset][:-1],trust_remote_code=True)[DATASET_HUGGINGFACE[each_dataset][-1]]
            elif DATASET_CONFIG[each_dataset].get('using_hf_api'):
                self.datasets[each_dataset] = load_dataset(
                    *DATASET_HUGGINGFACE[each_dataset][:-1], data_dir=DATASET_CONFIG[each_dataset]['data_dir'],trust_remote_code=True)[DATASET_HUGGINGFACE[each_dataset][-1]]
            elif DATASET_CONFIG[each_dataset].get('using_pandas'):
                if DATASET_CONFIG[each_dataset]['data_path'].split('.')[-1] == 'tsv':
                    self.datasets[each_dataset] = pd.read_csv(
                        DATASET_CONFIG[each_dataset]['data_path'], sep='\t')
                elif DATASET_CONFIG[each_dataset]['data_path'].split('.')[-1] == 'csv':
                    self.datasets[each_dataset] = pd.read_csv(
                        DATASET_CONFIG[each_dataset]['data_path'])
            elif DATASET_CONFIG[each_dataset].get('using_json'):
                self.datasets[each_dataset] = []
                if DATASET_CONFIG[each_dataset].get('raw_json'):
                    with open(DATASET_CONFIG[each_dataset]['data_path'], 'r', encoding='utf8') as f:
                        self.datasets[each_dataset] = json.load(f)
                else:
                    try:
                        json_file = json.load(
                            open(DATASET_CONFIG[each_dataset]['data_path'], 'r', encoding='utf8'))
                        for example in json_file:
                            self.datasets[each_dataset].append(example)
                    except:
                        with open(DATASET_CONFIG[each_dataset]['data_path'], 'r', encoding='utf8') as f:
                            for example in f:
                                self.datasets[each_dataset].append(
                                    json.loads(example))
            else:
                error('unable to locate raw dataset...')

    def process_stsb(self):
        output = []
        for example in tqdm(self.datasets['stsb'], desc=f'Constructing stsb'):
            text_a = example[DATASET_CONFIG['stsb']['text_a']]
            text_b = [example[DATASET_CONFIG['stsb']['text_b']]]
            text_c = []
            label = example[DATASET_CONFIG['stsb']['label']] / 5.0

            output.append({
                'text_a': text_a,
                'text_b': text_b,
                'text_c': text_c,
                'label': label
            })

        return output

    def init_qa_t5(self):
        from transformers import T5Tokenizer, T5ForConditionalGeneration
        if self.t5_qa is None:
            self.t5_tokenizer = T5Tokenizer.from_pretrained(
                "t5-base", model_max_length=800)
            self.t5_qa = T5ForConditionalGeneration.from_pretrained("t5-base")
            self.t5_qa.to('cuda:1')
            self.t5_qa.eval()

    @staticmethod
    def mask_answer(context, answers):
        answers = sorted(answers, key=len, reverse=True)
        for answer in answers:
            pattern = f'(?<![\w\\-\u2013]){re.escape(answer)}(?![\w\\-\u2013])'
            context = re.sub(pattern, '', context, flags=re.IGNORECASE)
        return context

    def generate_fake_answer(self, context, question, answers):
        self.init_qa_t5()

        context_no_answer = self.mask_answer(context, answers)

        input_ids = self.t5_tokenizer(
            f'question: {question} context: {context_no_answer}',
            return_tensors="pt",
            truncation='only_first'
        ).input_ids.to(self.t5_qa.device)

        outputs = self.t5_qa.generate(
            input_ids,
            max_new_tokens=40,
            remove_invalid_values=True
        )

        return self.t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

    def negative_sample_qa(self, samples, negative_sample_no_ans_only=True):
        outputs = []
        for context, question, answers in samples:
            if answers:
                outputs.append({
                    'text_a': context,
                    'text_b': [question],
                    'text_c': answers,
                    'label': 1
                })
            if not answers or not negative_sample_no_ans_only:
                fake_answer = self.generate_fake_answer(
                    context, question, answers)
                outputs.append({
                    'text_a': context,
                    'text_b': [question],
                    'text_c': [fake_answer],
                    'label': 0
                })

        return outputs

    def process_snli(self):
        output = []
        for example in tqdm(self.datasets['snli'], desc=f'Constructing snli'):
            text_a = example[DATASET_CONFIG['snli']['text_a']]
            text_b = [example[DATASET_CONFIG['snli']['text_b']]]
            text_c = []
            label = example[DATASET_CONFIG['snli']['label']]
            output.append({
                'text_a': text_a,
                'text_b': text_b,
                'text_c': text_c,
                'label': label
            })

        return output

    def process_anli(self):
        output = []
        for example in tqdm(self.datasets['anli'], desc=f'Constructing anli'):
            text_a = example[DATASET_CONFIG['anli']['text_a']]
            text_b = [example[DATASET_CONFIG['anli']['text_b']]]
            text_c = []
            label = example[DATASET_CONFIG['anli']['label']]
            output.append({
                'text_a': text_a,
                'text_b': text_b,
                'text_c': text_c,
                'label': label
            })
        return output
    
    def process_nli_fever(self):
        output = []
        for example in tqdm(self.datasets['nli_fever'], desc=f'Constructing nli_fever'):
            text_a = example[DATASET_CONFIG['nli_fever']['text_a']]
            text_b = [example[DATASET_CONFIG['nli_fever']['text_b']]]
            text_c = []
            label = example[DATASET_CONFIG['nli_fever']['label']]
            output.append({
                'text_a': text_a,
                'text_b': text_b,
                'text_c': text_c,
                'label': label
            })
        return output
        
    def process_sick(self):
        output = []
        for example in tqdm(self.datasets['sick'], desc=f'Constructing sick'):
            text_a = example[DATASET_CONFIG['sick']['text_a']]
            text_b = [example[DATASET_CONFIG['sick']['text_b']]]
            text_c = []
            label = example[DATASET_CONFIG['sick']['label']]
            output.append({
                'text_a': text_a,
                'text_b': text_b,
                'text_c': text_c,
                'label': label
            })
        return output

    def process_multi_nli(self):
        output = []
        for example in tqdm(self.datasets['multi_nli'], desc=f'Constructing multi_nli'):
            text_a = example[DATASET_CONFIG['multi_nli']['text_a']]
            text_b = [example[DATASET_CONFIG['multi_nli']['text_b']]]
            text_c = []
            label = example[DATASET_CONFIG['multi_nli']['label']]
            output.append({
                'text_a': text_a,
                'text_b': text_b,
                'text_c': text_c,
                'label': label
            })
        return output

    def process_vitaminc(self):
        label_map = {'SUPPORTS':0, 'NOT ENOUGH INFO':1, 'REFUTES':2}
        output = []
        for example in tqdm(self.datasets['vitaminc'], desc=f'Constructing vitaminc'):
            text_a = example[DATASET_CONFIG['vitaminc']['text_a']]
            text_b = [example[DATASET_CONFIG['vitaminc']['text_b']]]
            text_c = []
            label = example[DATASET_CONFIG['vitaminc']['label']]
            output.append({
                'text_a': text_a,
                'text_b': text_b,
                'text_c': text_c,
                'label': label_map[label]
            })
        return output
    
    def process_doc_nli(self):
        label_map = {'entailment':0, 'not_entailment':1}
        output = []
        for example in tqdm(self.datasets['doc_nli'], desc=f'Constructing doc_nli'):
            text_a = example[DATASET_CONFIG['doc_nli']['text_a']]
            text_b = [example[DATASET_CONFIG['doc_nli']['text_b']]]
            text_c = []
            label = example[DATASET_CONFIG['doc_nli']['label']]
            output.append({
                'text_a': text_a,
                'text_b': text_b,
                'text_c': text_c,
                'label': label_map[label]
            })
        return output
    
    def process_qqp(self):
        output = []
        for example in tqdm(self.datasets['qqp'], desc=f'Constructing qqp'):
            text_a = example[DATASET_CONFIG['qqp']['text_a']]
            text_b = [example[DATASET_CONFIG['qqp']['text_b']]]
            text_c = []
            label = example[DATASET_CONFIG['qqp']['label']]
            output.append({
                'text_a': text_a,
                'text_b': text_b,
                'text_c': text_c,
                'label': label
            })
        return output
    
    def process_paws(self):
        output = []
        for example in tqdm(self.datasets['paws'], desc=f'Constructing paws'):
            text_a = example[DATASET_CONFIG['paws']['text_a']]
            text_b = [example[DATASET_CONFIG['paws']['text_b']]]
            text_c = []
            label = example[DATASET_CONFIG['paws']['label']]
            output.append({
                'text_a': text_a,
                'text_b': text_b,
                'text_c': text_c,
                'label': label
            })
        return output
    
    def process_rufact(self):
        output = []
        for example in tqdm(self.datasets['rufact'], desc=f'Constructing rufact'):
            text_a = example[DATASET_CONFIG['rufact']['text_a']]
            text_b = [example[DATASET_CONFIG['rufact']['text_b']]]
            text_c = []
            label = example[DATASET_CONFIG['rufact']['label']]
            output.append({
                'text_a': text_a,
                'text_b': text_b,
                'text_c': text_c,
                'label': label
            })

        return output

    def process_ru_sts(self):
        output = []
        for example in tqdm(self.datasets['ru_sts'], desc=f'Constructing ru_sts'):
            text_a = example[DATASET_CONFIG['ru_sts']['text_a']]
            text_b = [example[DATASET_CONFIG['ru_sts']['text_b']]]
            text_c = []
            label = example[DATASET_CONFIG['ru_sts']['label']]
            output.append({
                'text_a': text_a,
                'text_b': text_b,
                'text_c': text_c,
                'label': label
            })
        return output
    
    def process_ms_marco(self):
        qa2d_generator = QA2D(batch_size=32, device=self.device)
        output = []
        correct_contexts = []
        correct_questions = []
        correct_answers = []

        wrong_contexts = []
        wrong_questions = []
        wrong_answers = []

        filtered_examples = []
        questions = []
        answers = []
        declaratives = []

        for example in tqdm(self.datasets['ms_marco'], desc=f'Collecting msmarco'):
            if sum(example['passages']['is_selected']) > 0:  # has answer
                questions.append(example['query'])
                if 'wellFormedAnswers' not in example.keys() or len(example['wellFormedAnswers']) == 0:
                    answers.append(example['answers'][0])
                else:
                    answers.append(example['wellFormedAnswers'][0])
                filtered_examples.append(example)
        
        for example in filtered_examples:
            for i, is_selected in enumerate(example['passages']['is_selected']):
                if is_selected == 1:
                    output.append({
                        'text_a': example['passages']['passage_text'][i],
                        'text_b': [example['query']],
                        'text_c': [],
                        'label': 1
                    })
                else:
                    output.append({
                        'text_a': example['passages']['passage_text'][i],
                        'text_b': [example['query']],
                        'text_c': [],
                        'label': 0
                    })
        return output
    
    def process_race(self):
        qa2d_generator = QA2D(batch_size=32, device=self.device)
        option_dict = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
        output = []

        correct_context = []
        correct_question = []
        correct_answer = []

        wrong_context = []
        wrong_question = []
        wrong_answer = []

        for example in tqdm(self.datasets['race'], desc=f'Constructing race'):
            text_a = example[DATASET_CONFIG['race']['text_a']]
            label = -1
            question = example[DATASET_CONFIG['race']['text_b'][0]]
            if "_" in question:
                answer_id = option_dict[example[DATASET_CONFIG['race']['label']]]
                for i, options in enumerate(example[DATASET_CONFIG['race']['text_b'][1]]):
                    if i == answer_id:
                        output.append({
                            'text_a': text_a,
                            'text_b': [' '.join(question.replace("_", " "+options+" ").split())],
                            'text_c': [],
                            'label': 1
                        })
                    else:
                        output.append({
                            'text_a': text_a,
                            'text_b': [' '.join(question.replace("_", " "+options+" ").split())],
                            'text_c': [],
                            'label': 0
                        })
            else:
                answer_id = option_dict[example[DATASET_CONFIG['race']['label']]]
                for i, options in enumerate(example[DATASET_CONFIG['race']['text_b'][1]]):
                    if i == answer_id:
                        output.append({
                                'text_a': text_a,
                                'text_b': [question],
                                'text_c': [options],
                                'label': 1
                            })
                    else:
                        output.append({
                                'text_a': text_a,
                                'text_b': [question],
                                'text_c': [options],
                                'label': 0
                            })

        return output


    def generate(self,path:str = './data/training'):
        if not os.path.exists(path):
            os.makedirs(path)
        for each_dataset in self.datasets:
            with open(f'{path}/{each_dataset}.json', 'w', encoding='utf8') as outfile:
                outfile.write("")
        for each_dataset in self.datasets:
            outputs = eval(f'self.process_{each_dataset}()')

            for each_output in outputs:
                dict_write_to_file = {
                    'task': DATASET_CONFIG[each_dataset]['task'],
                    'text_a': each_output['text_a'],  # string
                    # list of positive examples
                    'text_b': each_output['text_b'],
                    # list of negative examples
                    'text_c': each_output['text_c'],
                    # original label, if -1 only has positive pairs and negative pairs
                    'orig_label': each_output['label']
                }
                with open(f'{path}/{each_dataset}.json', 'a', encoding='utf8') as outfile:
                    json.dump(dict_write_to_file, outfile, ensure_ascii=False)
                    outfile.write('\n')

if __name__ == "__main__":
    random.seed(42)
    gen = DataGenerator(list(DATASET_CONFIG.keys()))
    gen.generate(path = './data/test')

Loading datasets:   0%|          | 0/13 [00:00<?, ?it/s]

Constructing snli:   0%|          | 0/1000 [00:00<?, ?it/s]

Constructing rufact:   0%|          | 0/1559 [00:00<?, ?it/s]

Constructing multi_nli:   0%|          | 0/1000 [00:00<?, ?it/s]

Constructing anli:   0%|          | 0/1000 [00:00<?, ?it/s]

...

Constructing qqp:   0%|          | 0/1000 [00:00<?, ?it/s]

Constructing paws:   0%|          | 0/1000 [00:00<?, ?it/s]

Constructing sick:   0%|          | 0/444 [00:00<?, ?it/s]

Constructing race:   0%|          | 0/1000 [00:00<?, ?it/s]

Constructing ru_sts:   0%|          | 0/783 [00:00<?, ?it/s]

In [10]:
! ls data/test

anli.json      multi_nli.json  qqp.json     ru_sts.json  vitaminc.json
doc_nli.json   nli_fever.json  race.json    sick.json
ms_marco.json  paws.json       rufact.json  snli.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [26]:
from scipy.stats import pearsonr, kendalltau, spearmanr
from sklearn.metrics import accuracy_score,precision_recall_fscore_support
from sklearn.metrics import roc_auc_score, mean_squared_error, r2_score
from sklearn.metrics import matthews_corrcoef
from datasets import load_dataset
from matplotlib import pyplot as plt
import json
import os
import numpy as np
from datasets import disable_progress_bars
from sklearn.metrics import (
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)


disable_progress_bars()

test_data_path = './data/test'
results_path = 'results'

def default_tri_grader(model,text_a,text_b,labels,name,plot_path = None):
    model.model.nlg_eval_mode = 'nli_sp'
    score = model.model.inference(text_a,text_b)
    tri_scores = score[2].argmax(axis=1)
    precision, recall, f1,_ = precision_recall_fscore_support(labels, tri_scores, average='micro')
    accuracy = accuracy_score(labels, tri_scores)
    mcc = matthews_corrcoef(labels, tri_scores)
    result = {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'accuracy': accuracy,
            'matthews_corrcoef': mcc,
    }
    if plot_path:
        #TODO add plots
        pass
    return result

def default_bin_grader(model,text_a,text_b,labels,name,threshold=0.5,plot_path = None):
    model.model.nlg_eval_mode = 'bin_sp'
    score = model.model.inference(premise=text_a, hypo=text_b)[1]
    
    precisions, recalls, thresholds = precision_recall_curve(labels, score)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    f1_scores = f1_scores[1:]
    best_idx    = np.argmax(f1_scores)
    best_thr    = thresholds[best_idx]

    threshold = best_thr
    tr_score = (score.numpy() > threshold).astype(int)
    
    
    precision, recall, f1, _ = precision_recall_fscore_support(labels, tr_score, average='binary')
    roc_auc = roc_auc_score(labels, score)
    mcc = matthews_corrcoef(labels, tr_score)
    results = {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'matthews_corrcoef': mcc,
        'best_thr':best_thr
    }
    if plot_path:
        plot_path = os.path.join(plot_path,name)
        os.makedirs(plot_path,exist_ok=True)
        # ── ROC Curve ──
        fpr, tpr, _ = roc_curve(labels, score)
        roc_auc = auc(fpr, tpr)
        
        plt.figure()
        plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.2f})")
        plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("Receiver Operating Characteristic (ROC) Curve")
        plt.legend(loc="best")
        
        plt.tight_layout()
        plt.savefig(os.path.join(plot_path,"roc_auc.png"))
        plt.close()
        
        
        # ── Precision–Recall Curve ──
        precision, recall, _ = precision_recall_curve(labels, score)
        avg_precision = average_precision_score(labels, score)
        
        plt.figure()
        plt.plot(recall, precision, label=f"PR curve (AP = {avg_precision:.2f})")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision–Recall (PR) Curve")
        plt.legend(loc="best")
        
        plt.tight_layout()
        plt.savefig(os.path.join(plot_path,"pr_curve.png"))
        plt.close()
    return results


def default_reg_grader(model,text_a,text_b,labels,name,plot_path = None):
    model.model.nlg_eval_mode = 'reg_sp'
    score = model.model.inference(premise=text_a, hypo=text_b)[0]
    mse = mean_squared_error(labels, score)
    r2 = r2_score(labels, score)
    results = {
        'mse': mse,
        'r2': r2,
    }
    if plot_path:
        #TODO add plots
        pass
    return results

GRADER_MAP = {
    'snli': default_tri_grader,
    'rufact': default_bin_grader,
    'multi_nli': default_tri_grader,
    'anli': default_tri_grader,
    'nli_fever': default_tri_grader,
    'vitaminc': default_tri_grader,
    'doc_nli': default_bin_grader,
    'qqp': default_bin_grader,
    'paws': default_bin_grader,
    'sick': default_reg_grader,
    'race': default_bin_grader, 
    'ms_marco': default_bin_grader,
    'ru_sts': default_reg_grader,
}

TEST_DATASETS = {}

bar = tqdm(GRADER_MAP.items())
for key,grader in bar:
    bar.set_description(f'Evaluating on {key}')
    TEST_DATASETS[key] = {}
    test_ds = TEST_DATASETS[key]
    test_ds['name'] = key
    
    test_ds['ds'] = load_dataset(
        "json",
        data_files=os.path.join(test_data_path,f"{test_ds['name']}.json"),  
        split='train'
    )
    assert len(set(test_ds['ds']['task']))==1, f'Something went wrong with tasks column for dataset {test_ds["name"]}, need 1 unique task, got {set(test_ds["ds"]["task"])}'
    test_ds['task'] = test_ds['ds'][0]['task']
    test_ds['ds'] = test_ds['ds'].map(
        lambda example: {"text_b_new": example["text_b"][0]}
    )
    test_ds['results'] = grader(model,text_a = test_ds['ds']['text_a'],
                                    text_b = test_ds['ds']['text_b_new'],
                                    labels= test_ds['ds']['orig_label'],name = test_ds['name'],plot_path=results_path)
    del test_ds['ds']

  0%|          | 0/13 [00:00<?, ?it/s]

TypeError: Object of type float32 is not JSON serializable

In [29]:
import json
import numpy as np

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

# … after building TEST_DATASETS …
with open(os.path.join(results_path, "result.json"), "w") as outfile:
    json.dump(TEST_DATASETS, outfile, indent=2, cls=NumpyEncoder)

In [30]:
TEST_DATASETS

{'snli': {'name': 'snli',
  'task': 'nli',
  'results': {'precision': 0.743,
   'recall': 0.743,
   'f1': 0.743,
   'accuracy': 0.743,
   'matthews_corrcoef': 0.6147572288928296}},
 'rufact': {'name': 'rufact',
  'task': 'paraphrase',
  'results': {'precision': 0.5907990314769975,
   'recall': 0.9081885856079405,
   'f1': 0.7158924205378973,
   'roc_auc': 0.7317751327197415,
   'matthews_corrcoef': 0.29060545120092524,
   'best_thr': 0.25944448}},
 'multi_nli': {'name': 'multi_nli',
  'task': 'nli',
  'results': {'precision': 0.674,
   'recall': 0.674,
   'f1': 0.674,
   'accuracy': 0.674,
   'matthews_corrcoef': 0.5112117456210372}},
 'anli': {'name': 'anli',
  'task': 'nli',
  'results': {'precision': 0.692,
   'recall': 0.692,
   'f1': 0.692,
   'accuracy': 0.692,
   'matthews_corrcoef': 0.5281489823713225}},
 'nli_fever': {'name': 'nli_fever',
  'task': 'fact_checking',
  'results': {'precision': 0.814,
   'recall': 0.814,
   'f1': 0.8140000000000001,
   'accuracy': 0.814,
   'matt

In [39]:
!zip results.zip results/ -r

updating: results/ (stored 0%)
  adding: results/doc_nli/ (stored 0%)
  adding: results/doc_nli/roc_auc.png (deflated 9%)
  adding: results/doc_nli/pr_curve.png (deflated 10%)
  adding: results/paws/ (stored 0%)
  adding: results/paws/roc_auc.png (deflated 8%)
  adding: results/paws/pr_curve.png (deflated 8%)
  adding: results/ms_marco/ (stored 0%)
  adding: results/ms_marco/roc_auc.png (deflated 6%)
  adding: results/ms_marco/pr_curve.png (deflated 13%)
  adding: results/race/ (stored 0%)
  adding: results/race/roc_auc.png (deflated 6%)
  adding: results/race/pr_curve.png (deflated 14%)
  adding: results/result.json (deflated 74%)
  adding: results/rufact/ (stored 0%)
  adding: results/rufact/roc_auc.png (deflated 6%)
  adding: results/rufact/pr_curve.png (deflated 8%)
  adding: results/qqp/ (stored 0%)
  adding: results/qqp/roc_auc.png (deflated 8%)
  adding: results/qqp/pr_curve.png (deflated 10%)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [40]:
! ls

data		  notebooks	     requirements.txt  src
evaluate.py	  outputname.tar.gz  result.json       train.py
generate_data.py  __pycache__	     results	       translate_datasets.py
LICENSE		  README.md	     results.zip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
